# MLP final: churn con presupuesto secuencial

Notebook reducido a la MLP de la segunda iteracion: `64 ReLU -> Dropout(0.2) -> 32 ReLU -> Dropout(0.1) -> 1`. Reproduce el pipeline, calibracion y regla de intervencion para obtener al menos **S/27 550** de presupuesto final.

## TL;DR

- Split 80/20 estratificado con semilla 42.
- One-hot para categoricas y StandardScaler para numericas.
- Cinco folds OOF y `EarlyStopping` por `val_loss`.
- Calibracion isotonic sobre OOF.
- Intervenir si `p(churn) > 0.10`, de mayor a menor probabilidad.
- El notebook falla si presupuesto final baja de S/27 550.

## 1. Configuracion, datos y preprocesamiento

La MLP usa `cuda:0`; se exige Tesla T4. `customerID` se conserva solo para desempatar el ranking.

In [ ]:
import copy
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
INITIAL_BUDGET = 1000
INTERVENTION_COST = 10
SUCCESS_REWARD = 100
THRESHOLD = INTERVENTION_COST / SUCCESS_REWARD
MINIMUM_FINAL_BUDGET = 27550
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert torch.cuda.is_available() and "T4" in gpu_name
DEVICE = torch.device("cuda:0")

data_path = next(Path("/kaggle/input").rglob("telco.csv"))
df = pd.read_csv(data_path)
assert len(df) == 7043 and df["customerID"].is_unique
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
assert (df["TotalCharges"].isna() == (df["tenure"] == 0)).all()
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(np.int8)

feature_columns = [c for c in df.columns if c not in {"customerID", "Churn", "ChurnFlag"}]
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_columns = [c for c in feature_columns if c not in numeric_columns]
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.20, stratify=df["ChurnFlag"], random_state=SEED)
assert (len(train_idx), len(test_idx)) == (5634, 1409)

X_train = df.loc[train_idx, feature_columns].reset_index(drop=True)
y_train = df.loc[train_idx, "ChurnFlag"].to_numpy()
X_test = df.loc[test_idx, feature_columns].reset_index(drop=True)
y_test = df.loc[test_idx, "ChurnFlag"].to_numpy()
ids_test = df.loc[test_idx, "customerID"].astype(str).to_numpy()

def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_columns),
        ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_columns),
    ], verbose_feature_names_out=False)

print({"gpu": gpu_name, "train": X_train.shape, "test": X_test.shape})

## 2. MLP y entrenamiento con EarlyStopping

La salida interna son logits para usar `BCEWithLogitsLoss`; `sigmoid` se aplica solo al producir probabilidades.

In [ ]:
class ChurnMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, features):
        return self.network(features).squeeze(1)


def predict_mlp(model, features):
    model.eval()
    tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return torch.sigmoid(model(tensor)).cpu().numpy()


def fit_mlp(X_fit, y_fit, X_valid=None, y_valid=None, epochs=200, patience=15):
    model = ChurnMLP(X_fit.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(
        TensorDataset(torch.as_tensor(X_fit, dtype=torch.float32), torch.as_tensor(y_fit, dtype=torch.float32)),
        batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(SEED), pin_memory=True,
    )
    best_loss, best_state, best_epoch, stale = np.inf, None, 0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(DEVICE, non_blocking=True), batch_y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        if X_valid is None:
            continue
        model.eval()
        with torch.no_grad():
            valid_loss = criterion(
                model(torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)),
                torch.as_tensor(y_valid, dtype=torch.float32, device=DEVICE),
            ).item()
        if valid_loss < best_loss - 1e-5:
            best_loss, best_state, best_epoch, stale = valid_loss, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            stale += 1
            if stale >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch or epochs

## 3. Predicciones OOF y calibracion

Cada score OOF proviene de un modelo que no entreno con esa fila. Isotonic se ajusta solo sobre esos scores de train.

In [ ]:
oof_scores = np.zeros(len(X_train))
best_epochs = []
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (fit_idx, valid_idx) in enumerate(folds.split(X_train, y_train), start=1):
    preprocessor = make_preprocessor()
    X_fit = preprocessor.fit_transform(X_train.iloc[fit_idx]).astype(np.float32)
    X_valid = preprocessor.transform(X_train.iloc[valid_idx]).astype(np.float32)
    model, best_epoch = fit_mlp(X_fit, y_train[fit_idx], X_valid, y_train[valid_idx])
    oof_scores[valid_idx] = predict_mlp(model, X_valid)
    best_epochs.append(best_epoch)
    print(f"fold={fold}/5 best_epoch={best_epoch}")

calibrator = IsotonicRegression(out_of_bounds="clip").fit(oof_scores, y_train)
oof_calibrated = calibrator.predict(oof_scores)
final_epochs = int(np.median(best_epochs))
print({
    "final_epochs": final_epochs,
    "roc_auc_oof": round(roc_auc_score(y_train, oof_calibrated), 4),
    "pr_auc_oof": round(average_precision_score(y_train, oof_calibrated), 4),
    "brier_oof": round(brier_score_loss(y_train, oof_calibrated), 4),
})

## 4. Ajuste final, decision y comprobacion

Se ajusta una MLP final con la mediana de epochs OOF. Antes de revelar etiquetas test se congela lista `p(churn) > 0.10`, ordenada por score y luego `customerID`.

In [ ]:
final_preprocessor = make_preprocessor()
X_train_final = final_preprocessor.fit_transform(X_train).astype(np.float32)
X_test_final = final_preprocessor.transform(X_test).astype(np.float32)
final_model, _ = fit_mlp(X_train_final, y_train, epochs=final_epochs)
test_scores = calibrator.predict(predict_mlp(final_model, X_test_final))

eligible = np.flatnonzero(test_scores > THRESHOLD)
ordered_indices = eligible[np.lexsort((ids_test[eligible], -test_scores[eligible]))]
assert (test_scores[ordered_indices] > THRESHOLD).all()
assert np.all(test_scores[ordered_indices][:-1] >= test_scores[ordered_indices][1:])

budget = INITIAL_BUDGET
ledger_rows = []
for rank, row_index in enumerate(ordered_indices, start=1):
    if budget < INTERVENTION_COST:
        break
    before = budget
    budget -= INTERVENTION_COST
    success = int(y_test[row_index]) == 1
    if success:
        budget += SUCCESS_REWARD
    ledger_rows.append({
        "rank": rank,
        "customerID": ids_test[row_index],
        "calibrated_probability": test_scores[row_index],
        "actual_churn": int(y_test[row_index]),
        "budget_before": before,
        "success": success,
        "budget_after": budget,
        "cumulative_profit": budget - INITIAL_BUDGET,
    })

ledger = pd.DataFrame(ledger_rows)
profit = budget - INITIAL_BUDGET
successes = int(ledger["success"].sum())
assert profit == successes * SUCCESS_REWARD - len(ledger) * INTERVENTION_COST
assert budget >= MINIMUM_FINAL_BUDGET, f"Presupuesto minimo {MINIMUM_FINAL_BUDGET}; obtenido {budget}"

summary = {
    "status": "complete", "model": "MLP", "gpu": gpu_name, "seed": SEED,
    "architecture": "Dense(64)-ReLU-Dropout(0.2)-Dense(32)-ReLU-Dropout(0.1)-Dense(1)",
    "calibration": "isotonic sobre scores OOF", "final_epochs": final_epochs,
    "initial_budget": INITIAL_BUDGET, "minimum_expected_budget": MINIMUM_FINAL_BUDGET,
    "final_budget": int(budget), "profit": int(profit), "planned_interventions": int(len(ordered_indices)),
    "executed_interventions": int(len(ledger)), "successes": successes, "failures": int(len(ledger)-successes),
    "stopped_for_budget": bool(len(ledger) < len(ordered_indices)),
    "roc_auc_test": float(roc_auc_score(y_test, test_scores)),
    "pr_auc_test": float(average_precision_score(y_test, test_scores)),
    "brier_test": float(brier_score_loss(y_test, test_scores)),
}
ledger.to_csv(OUTPUT_DIR / "intervention_ledger.csv", index=False)
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Takeaways

Ejecucion valida solo si confirma un presupuesto final de al menos S/27 550. La MLP usa CUDA; preprocesamiento, calibracion y simulacion son operaciones pequenas de CPU.